# Day 12 — Solution: The Weekend Effect

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    px = get_prices("SPY", start="1953-01-01")
else:
    px = synthetic_prices(n_days=15000, n_assets=1, seed=12, mu=0.0)
r = px.iloc[:, 0].pct_change().dropna()
DOW = r.index.dayofweek          # 0 = Monday

## E1 — the table

In [ ]:
g = r.groupby(DOW)
tab = pd.DataFrame({"mean(bp)": g.mean()*1e4, "sd(bp)": g.std()*1e4,
                    "n": g.size()})
tab["t"] = tab["mean(bp)"] / (tab["sd(bp)"] / np.sqrt(tab["n"]))
print(tab.round(2))

**Expected reasoning.** On real SPY the table is roughly flat with
Monday the most negative day in most samples (−1 to −3bp, t usually
|t| < 2 over the full 1953–present span); Friday is often the most
positive. On the synthetic (null) world the table is *exactly* the
reference for how noise looks at this sample size: the most
extreme day will have |t| ~ 1.5–2.5 by chance alone. Read the real
table against the synthetic one — that comparison is the whole audit.

## E2 — the contrast and the family

In [ ]:
mon = r[DOW == 0]; oth = r[DOW != 0]
se = np.sqrt(mon.var()/len(mon) + oth.var()/len(oth))
t_con = (mon.mean() - oth.mean()) / se
p_con = 2*stats.t.sf(abs(t_con), np.inf)  # df huge; Welch df nearly identical

days = {d: r[DOW == d] for d in range(5)}
p_raw = {d: 2*stats.t.sf(abs(x.mean()/(x.std()/np.sqrt(len(x)))), len(x)-1)
         for d, x in days.items()}
p_arr = np.array(list(p_raw.values()))
bonf = {d: "sig" if p < 0.05/5 else "ns" for d, p in p_raw.items()}
# BH
order = np.argsort(p_arr); k = 0
for i in range(len(p_arr)):
    if p_arr[order[i]] <= (i+1)*0.05/5:
        k = i + 1
bh_sig = [d for d, p in p_raw.items() if p <= 0.05] if k else []
print(f"Monday vs others: t = {t_con:.2f}, p = {p_con:.4f}")
print("raw p's:", {d: round(p, 4) for d, p in p_raw.items()})
print(f"Bonferroni (alpha/5 = 0.01): {bonf}   BH@.05 rejects: {k}")

**Expected reasoning.** Real data: t_con typically −1 to −2 — inside
the noise band; Bonferroni rarely rejects any day over the full span;
the pre-1980 subsample (E3) is where French's effect lived. Synthetic
data: the placebo — expect zero Bonferroni rejections most of the time
but *sometimes* one (1 − 0.95^5 ≈ 23% of noise studies show a raw
5% "day"; after Bonferroni, ~5%). **The placebo quantifies the audit's
own false-positive rate** — the number a paper should report.

## E3 — stability

In [ ]:
def blocks(x, k):
    n = len(x)
    return [x[i*n//k:(i+1)*n//k] for i in range(k)]

for name, blk in [("full sample", r)] + \
                 [(f"block {i+1}/4", b) for i, b in enumerate(blocks(r, 4))]:
    dw = blk.index.dayofweek
    m, o = blk[dw == 0], blk[dw != 0]
    t = (m.mean() - o.mean())/np.sqrt(m.var()/len(m) + o.var()/len(o))
    print(f"{name:12s} n={len(blk):5d}  Monday−others t = {t:+.2f}")

**Expected reasoning.** Real data: the classic arc — the contrast is
weakest in recent blocks and (if your sample starts early) strongest in
the first block: the effect French saw did not survive its own
subsequent history. Synthetic data: four blocks "discover" a significant
day in roughly 20% of them (4 × 5% per block, minus overlap) — noise
produces the *pattern* of an effect in some sub-periods and its
absence in others, which is exactly why "it was there before, not
after" is not evidence of a real effect without the placebo.

## E4 — the tradability gate

In [ ]:
edge = mon.mean()                      # per Monday (short earns -edge if edge<0)
COST, EV = 0.0010, 52
gross_ann = -edge * EV if edge < 0 else -abs(edge) * EV
net_ann = gross_ann - COST * EV
breakeven = COST
print(f"per-event edge: {edge*1e4:+.2f} bp | gross ann: {gross_ann*1e4:+.0f} bp/yr")
print(f"net of 10bp/event: {net_ann*1e4:+.0f} bp/yr | breakeven edge: {breakeven*1e4:.0f} bp/event")
print("verdict:", "LIVE" if abs(edge) > COST else "DEAD at 10bp round-trip")

**Expected reasoning.** A realistic |edge| of 1–3bp/event is 2–5×
below the 10bp break-even: **the strategy is dead at any realistic
cost before we finish the sentence.** This is the course's central
pattern in miniature: a statistically visible effect (t ≈ 2) that
dies at the cost gate (04.17). If your sample shows |edge| > 10bp,
stop and suspect a data artifact (05.1's domain: an adjustment or a
timestamp, not an alpha).

## E5 — the mini-report (exemplar, real-data flavor)

**Hypothesis:** H2 — Monday mean return < other-days mean return.
**Data:** SPY adjusted daily, 1953–present, ~18,400 observations;
universe = one index (limits generality; the original was the NYSE).
**Method:** one-sample t per day (vs 0); Welch contrast Monday vs
other-4 pooled; family = 5 day-tests, Bonferroni and BH(0.05); 4
non-overlapping sub-periods for stability; Welch SE throughout.
**Result:** Monday mean −2.xbp, t ≈ −1.x; contrast t ≈ −1.x; raw
p's all > 0.05/5 — the family verdict is null on the full sample.
**Stability:** the contrast strengthens in the earliest block and
weakens/vanishes in later blocks (the French sample's arc).
**Cost:** 10bp round-trip vs ~2bp per event → net ≈ −496bp/yr;
breakeven ≈ 10bp/event, ~5× the observed edge.
**Verdict:** on this sample the weekend effect is statistically
marginal, time-varying, and economically negative after costs;
consistent with the published non-replication record — indistinguishable
from the synthetic placebo except in the earliest block, where the
residual is a hypothesis for a different sample, not a strategy.